# 🧬 Drug-Disease HGT — Train ALL Datasets (B / C / F)

Notebook này train cả **Original** và **Improved** model trên 3 dataset.

**Output:** file `best_model.pth` cho từng model × dataset → download về laptop.

| Dataset | Original target AUC | Improved target AUC |
|---------|--------------------|-----------------|
| B-dataset | ~0.900 | ~0.920 |
| C-dataset | ~0.880 | ~0.905 |
| F-dataset | ~0.870 | ~0.895 |

---
**Sau khi chạy xong**, download file từ `/kaggle/working/checkpoints/` và đặt vào:
```
Colab_V4/Result/original/<dataset>/best_model.pth
Colab_V4/Result/improved/<dataset>/best_model.pth
```

In [ ]:
# ── Cell 1: Setup môi trường ──────────────────────────────
import subprocess, sys, os

# Clone repo
if not os.path.exists('/kaggle/working/Colab_V4'):
    subprocess.run(['git', 'clone', 'https://github.com/HuynhTuanKiet05/Colab_V4.git',
                    '/kaggle/working/Colab_V4'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/Colab_V4', 'pull'], check=True)

os.chdir('/kaggle/working/Colab_V4')
sys.path.insert(0, '/kaggle/working/Colab_V4')
sys.path.insert(0, '/kaggle/working/Colab_V4/python_api')

print('Repo ready!')

In [ ]:
# ── Cell 2: Cài thư viện ──────────────────────────────────
# Kaggle đã có torch, dgl — chỉ cần thêm:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'dgl', '-f', 'https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'networkx', 'scipy', 'scikit-learn'], check=True)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ── Cell 3: Import tất cả module cần thiết ────────────────
import gc, timeit
from pathlib import Path
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

REPO    = Path('/kaggle/working/Colab_V4')
OUTPUT  = Path('/kaggle/working/checkpoints')
OUTPUT.mkdir(exist_ok=True)

print('Imports OK')

In [ ]:
# ── Cell 4: Hyperparameter presets ───────────────────────
IMPROVED_PRESETS = {
    'B-dataset': dict(lr=1e-4, weight_decay=1e-3, neighbor=3,
                      gt_out_dim=512, hgt_layer=2, hgt_in_dim=512,
                      hgt_head=8, hgt_head_dim=64, topo_hidden=192,
                      similarity_view_mode='consensus', positive_weight_mode='none'),
    'C-dataset': dict(lr=1e-4, weight_decay=1e-3, neighbor=5,
                      gt_out_dim=256, hgt_layer=2, hgt_in_dim=256,
                      hgt_head=8, hgt_head_dim=32, topo_hidden=128,
                      similarity_view_mode='consensus', positive_weight_mode='none'),
    'F-dataset': dict(lr=8e-5, weight_decay=1e-3, neighbor=10,
                      gt_out_dim=384, hgt_layer=3, hgt_in_dim=384,
                      hgt_head=8, hgt_head_dim=48, topo_hidden=192,
                      similarity_view_mode='multi', positive_weight_mode='global_log'),
}

ORIGINAL_PRESET = dict(lr=1e-4, weight_decay=1e-3, neighbor=20,
                       gt_out_dim=200, hgt_out_dim=200, hgt_layer=2,
                       hgt_in_dim=64, hgt_head=8, hgt_head_dim=25,
                       gt_layer=2, gt_head=2, tr_layer=2, tr_head=4)

def make_args(dataset, data_dir, preset, version='improved'):
    class A: pass
    a = A()
    a.dataset = dataset
    a.data_dir = str(data_dir)
    a.k_fold = 10
    a.negative_rate = 1.0
    a.dropout = 0.2
    a.random_seed = 1234
    a.gt_layer = preset.get('gt_layer', 2)
    a.gt_head = preset.get('gt_head', 2)
    a.gt_out_dim = preset['gt_out_dim']
    a.hgt_layer = preset['hgt_layer']
    a.hgt_head = preset.get('hgt_head', 8)
    a.hgt_head_dim = preset['hgt_head_dim']
    a.hgt_in_dim = preset['hgt_in_dim']
    a.hgt_out_dim = preset.get('hgt_out_dim', preset['gt_out_dim'])
    a.tr_layer = preset.get('tr_layer', 2)
    a.tr_head = preset.get('tr_head', 4)
    a.neighbor = preset['neighbor']
    if version == 'improved':
        a.assoc_backbone = 'vanilla_hgt'
        a.fusion_mode = 'mva'
        a.pair_mode = 'mul_mlp'
        a.gate_mode = 'vector'
        a.gate_bias_init = -2.0
        a.temperature = 0.5
        a.topo_hidden = preset.get('topo_hidden', 128)
        a.topo_feat_dim = 7
        a.similarity_view_mode = preset.get('similarity_view_mode', 'consensus')
        a.positive_weight_mode = preset.get('positive_weight_mode', 'none')
        a.device = str(device)
    return a

print('Presets ready')

In [ ]:
# ── Cell 5: Hàm train ORIGINAL model ─────────────────────
from metric import get_metric

def train_original(dataset, epochs=1000, k_fold=10):
    from AMDGT_original.data_preprocess import (
        get_data, data_processing, k_fold as make_kfold,
        dgl_similarity_graph, dgl_heterograph,
    )
    from AMDGT_original.model.AMNTDDA import AMNTDDA

    print(f'\n{"="*60}')
    print(f'ORIGINAL | {dataset} | {epochs} epochs | {k_fold} folds')
    print(f'{"="*60}')

    data_dir = REPO / 'AMDGT_original' / 'data' / dataset
    args = make_args(dataset, data_dir, ORIGINAL_PRESET, 'original')
    args.k_fold = k_fold

    data = get_data(args)
    args.drug_number = data['drug_number']
    args.disease_number = data['disease_number']
    args.protein_number = data['protein_number']
    data = data_processing(data, args)
    data = make_kfold(data, args)

    drdr_g, didi_g, data = dgl_similarity_graph(data, args)
    drdr_g = drdr_g.to(device); didi_g = didi_g.to(device)
    df = torch.FloatTensor(data['drugfeature']).to(device)
    dsf = torch.FloatTensor(data['diseasefeature']).to(device)
    pf = torch.FloatTensor(data['proteinfeature']).to(device)

    best_auc_overall = -1.0; best_state = None; aucs = []

    for fold in range(k_fold):
        print(f'\n--- Fold {fold} ---')
        model = AMNTDDA(args).to(device)
        opt = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        crit = nn.CrossEntropyLoss()
        xt = torch.LongTensor(data['X_train'][fold]).to(device)
        yt = torch.LongTensor(data['Y_train'][fold]).to(device).flatten()
        xv = torch.LongTensor(data['X_test'][fold]).to(device)
        yv = data['Y_test'][fold].flatten()
        drdipr_g, data = dgl_heterograph(data, data['X_train'][fold], args)
        drdipr_g = drdipr_g.to(device)

        best_fold = -1.0; no_imp = 0; t0 = timeit.default_timer()
        for ep in range(epochs):
            model.train()
            _, sc = model(drdr_g, didi_g, drdipr_g, df, dsf, pf, xt)
            loss = crit(sc, yt)
            opt.zero_grad(); loss.backward(); opt.step()

            if (ep+1) % 50 == 0 or ep == epochs-1:
                model.eval()
                with torch.no_grad():
                    _, ts = model(drdr_g, didi_g, drdipr_g, df, dsf, pf, xv)
                prob = F.softmax(ts, dim=-1)[:,1].cpu().numpy()
                pred = torch.argmax(ts,dim=-1).cpu().numpy()
                auc, *_ = get_metric(yv, pred, prob)
                elapsed = timeit.default_timer() - t0
                print(f'  Ep {ep+1:4d} | AUC={auc:.4f} | {elapsed:.1f}s')
                if auc > best_fold:
                    best_fold = auc; no_imp = 0
                    if auc > best_auc_overall:
                        best_auc_overall = auc
                        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                else:
                    no_imp += 50
                if no_imp >= 300: print(f'  Early stop'); break

        aucs.append(best_fold)
        print(f'  Fold {fold} AUC: {best_fold:.4f}')
        del model, opt, drdipr_g; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nOriginal {dataset} | Mean AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
    out = OUTPUT / 'original' / dataset
    out.mkdir(parents=True, exist_ok=True)
    path = out / 'best_model.pth'
    torch.save({'model_state_dict': best_state, 'auc': best_auc_overall,
                'dataset': dataset, 'version': 'original', 'mean_auc': float(np.mean(aucs))}, path)
    print(f'Saved: {path}')
    return path

print('train_original() defined')

In [ ]:
# ── Cell 6: Hàm train IMPROVED model ─────────────────────
def train_improved(dataset, epochs=1000, k_fold=10):
    from data_preprocess_improved import (
        get_data, data_processing, k_fold as make_kfold,
        dgl_similarity_graph, dgl_heterograph, dgl_similarity_view_graphs,
    )
    from model.improved.improved_model import AMNTDDA
    from topology_features import extract_topology_features

    print(f'\n{"="*60}')
    print(f'IMPROVED | {dataset} | {epochs} epochs | {k_fold} folds')
    print(f'{"="*60}')

    preset = IMPROVED_PRESETS[dataset]
    data_dir = REPO / 'AMDGT_original' / 'data' / dataset
    args = make_args(dataset, data_dir, preset, 'improved')
    args.k_fold = k_fold

    data = get_data(args)
    args.drug_number = data['drug_number']
    args.disease_number = data['disease_number']
    args.protein_number = data['protein_number']
    data = data_processing(data, args)
    data = make_kfold(data, args)

    if args.similarity_view_mode == 'multi':
        drdr_g, didi_g, data = dgl_similarity_view_graphs(data, args)
        drdr_g = {k: v.to(device) for k,v in drdr_g.items()}
        didi_g = {k: v.to(device) for k,v in didi_g.items()}
    else:
        drdr_g, didi_g, data = dgl_similarity_graph(data, args)
        drdr_g = drdr_g.to(device); didi_g = didi_g.to(device)

    dt, dst = extract_topology_features(data, args)
    dt = dt.to(device); dst = dst.to(device)
    df  = torch.tensor(data['drugfeature'],    dtype=torch.float32).to(device)
    dsf = torch.tensor(data['diseasefeature'], dtype=torch.float32).to(device)
    pf  = torch.tensor(data['proteinfeature'], dtype=torch.float32).to(device)

    best_auc_overall = -1.0; best_state = None; aucs = []

    for fold in range(k_fold):
        print(f'\n--- Fold {fold} ---')
        model = AMNTDDA(args).to(device)
        opt = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        sched = ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=30, min_lr=1e-6)
        crit = nn.CrossEntropyLoss(label_smoothing=0.01)
        xt = torch.tensor(data['X_train'][fold], dtype=torch.long, device=device)
        yt = torch.tensor(data['Y_train'][fold], dtype=torch.long, device=device).flatten()
        xv = torch.tensor(data['X_test'][fold],  dtype=torch.long, device=device)
        yv = data['Y_test'][fold].flatten()
        res = dgl_heterograph(data, data['X_train'][fold], args)
        drdipr_g = res[0].to(device)

        best_fold = -1.0; no_imp = 0; t0 = timeit.default_timer()
        for ep in range(epochs):
            model.train()
            _, sc, aux = model(drdr_g, didi_g, drdipr_g, df, dsf, pf, xt,
                               drug_topo_feat=dt, disease_topo_feat=dst, return_aux=True)
            cl = aux.get('contrastive', sc.new_tensor(0.0))
            loss = crit(sc, yt) + 0.1*cl
            opt.zero_grad(); loss.backward(); opt.step()

            if (ep+1) % 50 == 0 or ep == epochs-1:
                model.eval()
                with torch.no_grad():
                    _, ts, _ = model(drdr_g, didi_g, drdipr_g, df, dsf, pf, xv,
                                     drug_topo_feat=dt, disease_topo_feat=dst,
                                     return_diagnostics=True)
                prob = F.softmax(ts, dim=-1)[:,1].cpu().numpy()
                pred = torch.argmax(ts,dim=-1).cpu().numpy()
                auc, *_ = get_metric(yv, pred, prob)
                sched.step(auc)
                elapsed = timeit.default_timer() - t0
                print(f'  Ep {ep+1:4d} | AUC={auc:.4f} | loss={loss.item():.4f} | {elapsed:.1f}s')
                if auc > best_fold:
                    best_fold = auc; no_imp = 0
                    if auc > best_auc_overall:
                        best_auc_overall = auc
                        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                else:
                    no_imp += 50
                if no_imp >= 250: print('  Early stop'); break

        aucs.append(best_fold)
        print(f'  Fold {fold} AUC: {best_fold:.4f}')
        del model, opt, sched, drdipr_g; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nImproved {dataset} | Mean AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
    out = OUTPUT / 'improved' / dataset
    out.mkdir(parents=True, exist_ok=True)
    path = out / 'best_model.pth'
    torch.save({'model_state_dict': best_state, 'auc': best_auc_overall,
                'dataset': dataset, 'version': 'improved', 'mean_auc': float(np.mean(aucs))}, path)
    print(f'Saved: {path}')
    return path

print('train_improved() defined')

In [ ]:
# ── Cell 7: Train B-dataset ───────────────────────────────
# Thời gian ước tính: ~20-40 phút (GPU T4)
train_original('B-dataset', epochs=1000, k_fold=10)
train_improved('B-dataset', epochs=1000, k_fold=10)

In [ ]:
# ── Cell 8: Train C-dataset ───────────────────────────────
# Thời gian ước tính: ~30-60 phút (GPU T4)
train_original('C-dataset', epochs=1000, k_fold=10)
train_improved('C-dataset', epochs=1000, k_fold=10)

In [ ]:
# ── Cell 9: Train F-dataset ───────────────────────────────
# Thời gian ước tính: ~40-80 phút (GPU T4)
train_original('F-dataset', epochs=1000, k_fold=10)
train_improved('F-dataset', epochs=1000, k_fold=10)

In [ ]:
# ── Cell 10: Kiểm tra kết quả và hướng dẫn download ──────
import os
print('=== CHECKPOINTS DA TAO ===')
for p in sorted(Path('/kaggle/working/checkpoints').rglob('*.pth')):
    ck = torch.load(p, map_location='cpu', weights_only=False)
    auc = ck.get('auc', ck.get('mean_auc', '?'))
    size_mb = p.stat().st_size / 1024**2
    print(f'  {p.relative_to("/kaggle/working/checkpoints")} | AUC={auc:.4f} | {size_mb:.1f} MB')

print()
print('=== HUONG DAN DOWNLOAD ===')
print('1. Click Output (panel ben phai Kaggle)')
print('2. Download tung file best_model.pth')
print('3. Dat vao may tinh:')
print('   Result/original/B-dataset/best_model.pth')
print('   Result/original/C-dataset/best_model.pth')
print('   Result/original/F-dataset/best_model.pth')
print('   Result/improved/B-dataset/best_model.pth')
print('   Result/improved/C-dataset/best_model.pth')
print('   Result/improved/F-dataset/best_model.pth')
print('4. Chay restart_api.bat de uvicorn load model moi')